In [1]:
import os
from pathlib import Path
from ures.files import filter_files
from ures.string import format_memory
from perf_estimator.estimator import Estimator
from perf_estimator.dataset import image_dataset
from experiments.snapshot import SnapshotAnalyser



In [2]:
# Setup Basic Global Variables
root_dir = Path("/Users/jiaboshi/Documents/101-Data/002-xMem-LLM")
root_dir = Path("/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/")
pytorch_dir = root_dir / "001-PyTorch"
huggingface_dir = root_dir / "002-HuggingFace"

pytorch_dir_name_format = "recurrence-{}-SGD-{}-1"
huggingface_dir_name_format_xmem = "{}-{}-xMem"
huggingface_dir_name_format_llm = "{}-{}-LLM"
huggingface_dir_name_format_cuda = "{}-{}-CUDA"

In [3]:
model = "ConvNeXtTiny"
batch_size = "170"
max_gpu_memory = 8

In [4]:
def get_xmem_memory(profiler_file, batch, max_in_gb):
    estimator = Estimator(
        dataloader=image_dataset(batch=int(batch)),
        profiler_file=profiler_file,
        max_gpu_memory_in_gb=max_in_gb
    )
    my_result, _ = estimator.estimate()
    return max(my_result._trace.max_segment_changes)


def get_ground_value_from_snapshot(snapshot_file: str) -> dict:
    _snapshot = SnapshotAnalyser(snapshot_file)
    return _snapshot.gpu_and_segment_in_same_time_length()



In [5]:
from typing import Union
def get_all_memory_information(model_name: str, batch: Union[str, int]) -> dict:
    batch = str(batch)
    # Get PyTorch Data
    torch_data_dir = pytorch_dir.joinpath(pytorch_dir_name_format.format(model_name, batch))
    all_torch_dirs = os.listdir(torch_data_dir)
    all_torch_dirs = [torch_data_dir.joinpath(d) for d in all_torch_dirs if str(d).startswith(".") is False]
    dirs_sorted = sorted(all_torch_dirs, key=lambda d: d.stat().st_ctime)
    torch_snapshot_file = filter_files(".pickle", dirs_sorted[0], fuzz=True)[-1]
    torch_profiler_file = filter_files(".pt.trace.json", dirs_sorted[0], fuzz=True)[-1]

    # Get HuggingFace Data
    huggingface_dir_xmem = huggingface_dir.joinpath(huggingface_dir_name_format_xmem.format(model_name, batch))
    huggingface_dir_cuda = huggingface_dir.joinpath(huggingface_dir_name_format_cuda.format(model_name, batch))
    huggingface_dir_llm = huggingface_dir.joinpath(huggingface_dir_name_format_llm.format(model_name, batch))
    huggingface_profiler_file_xmem = filter_files(".pt.trace.json", huggingface_dir_xmem, fuzz=True)[-1]
    huggingface_profiler_file_llm = filter_files(".pt.trace.json", huggingface_dir_llm, fuzz=True)[-1]
    huggingface_snapshot_file_xmem = filter_files(".pickle", huggingface_dir_cuda, fuzz=True)[-1]

    # Estimate memory
    huggingface_memory_llm = get_xmem_memory(
        profiler_file=huggingface_profiler_file_llm,
        batch=batch_size,
        max_in_gb=max_gpu_memory
    )
    huggingface_snapshot_memory_xmen = max(get_ground_value_from_snapshot(huggingface_snapshot_file_xmem)['seg'])
    huggingface_memory_diff = huggingface_memory_llm - huggingface_snapshot_memory_xmen

    paper_memory_result = get_xmem_memory(
        profiler_file=torch_profiler_file,
        batch=batch_size,
        max_in_gb=max_gpu_memory
    )
    paper_snapshot_result = max(get_ground_value_from_snapshot(torch_snapshot_file)['seg'])
    paper_memory_diff = paper_memory_result - paper_snapshot_result

    return {
        "torch": {
            "est": paper_memory_result,
            "ground": paper_snapshot_result,
            "diff": paper_memory_diff
        },
        "huggingface": {
            "est": huggingface_memory_llm,
            "ground": huggingface_snapshot_memory_xmen,
            "diff": huggingface_memory_diff
        }
    }


In [6]:
models = ["ConvNeXtTiny", "ResNet50", "VGG16"]
batch = range(10, 570, 40)

In [ ]:
result_list = []
for m in models:
    for b in batch:
        _r = get_all_memory_information(m, b)
        _r.update({
            "model": m,
            "batch": b
        })
        result_list.append(_r)

In [ ]:
import pandas as pd
df = pd.json_normalize(result_list)

In [ ]:
df


# Analysis of Shopshot Data between PyTorch and HuggingFace


In [4]:
model_name = model
batch = batch_size
torch_data_dir = pytorch_dir.joinpath(pytorch_dir_name_format.format(model_name, batch))
all_torch_dirs = os.listdir(torch_data_dir)
all_torch_dirs = [torch_data_dir.joinpath(d) for d in all_torch_dirs if str(d).startswith(".") is False and str(d).startswith("@") is False]
dirs_sorted = sorted(all_torch_dirs, key=lambda d: d.stat().st_ctime)
dirs_sorted

[PosixPath('/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/001-PyTorch/recurrence-ConvNeXtTiny-SGD-170-1/20241126-024446-7ed9'),
 PosixPath('/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/001-PyTorch/recurrence-ConvNeXtTiny-SGD-170-1/20241126-024218-1b65'),
 PosixPath('/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/001-PyTorch/recurrence-ConvNeXtTiny-SGD-170-1/20241126-024332-ee54')]

In [5]:
torch_snapshot_file = Path(filter_files(".pickle", dirs_sorted[0], fuzz=True)[0])
torch_profiler_file = Path(filter_files(".pt.trace.json", dirs_sorted[0], fuzz=True)[-1])


In [6]:
# Get HuggingFace Data
huggingface_dir_xmem = huggingface_dir.joinpath(huggingface_dir_name_format_xmem.format(model_name, batch))
huggingface_dir_cuda = huggingface_dir.joinpath(huggingface_dir_name_format_cuda.format(model_name, batch))
huggingface_dir_llm = huggingface_dir.joinpath(huggingface_dir_name_format_llm.format(model_name, batch))
huggingface_profiler_file_xmem = filter_files(".pt.trace.json", huggingface_dir_xmem, fuzz=True)[-1]
huggingface_profiler_file_llm = filter_files(".pt.trace.json", huggingface_dir_llm, fuzz=True)[-1]
huggingface_snapshot_file_xmem = Path(filter_files(".pickle", huggingface_dir_cuda, fuzz=True)[-1])

torch_snapshot_file

PosixPath('/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/001-PyTorch/recurrence-ConvNeXtTiny-SGD-170-1/20241126-024446-7ed9/results/snapshot/snapshot_result-1732589151.pickle')

In [7]:
torch_snapshot = SnapshotAnalyser(torch_snapshot_file)
hugging_snapshot = SnapshotAnalyser(huggingface_snapshot_file_xmem)

In [11]:
torch_snapshot.save_json(torch_snapshot_file.parent.joinpath("snapshot.json"))
torch_blocks = torch_snapshot.breakdown_into_multiple_sections()
torch_snapshot_file

PosixPath('/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/001-PyTorch/recurrence-ConvNeXtTiny-SGD-130-1/20241126-010200-7c39/results/snapshot/snapshot_result-1732582988.pickle')

In [15]:
hugging_snapshot.save_json(huggingface_snapshot_file_xmem.parent.joinpath("snapshot.json"))
hugging_blocks = hugging_snapshot.breakdown_into_multiple_sections()

IndexError: list index out of range

In [10]:
xx = torch_snapshot.group_by_file("/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/transformers")

Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activities: 0
Number of activi

In [9]:
xx

{'2171': [clamp(False|True|137059945296384): bytes: 4, start: 1739489917389521, stop: 1739489917389605]}